# Day 1: Golf Swing Processing - Frame Extraction, Smoothing & Normalization

This notebook demonstrates Phase 1 of the Golf Swing Analysis pipeline. We will:
1. Load a raw golf swing video clip.
2. Extract pose landmarks using Google's **MediaPipe Tasks Pose Landmarker**.
3. Apply a **Savitzky-Golay filter** to smooth out coordinate micro-jitters.
4. Center coordinates relative to the **mid-hip** and scale them by the **torso scale factor** (calculated from the first 5 frames).
5. Save the output to a master CSV DataFrame.
6. Generate visual trajectories and skeleton overlay/stick figure animations.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src to python path
sys.path.append(os.path.abspath('../'))
from src.data_processor import GolfVideoProcessor

## 1. Run the Video Processor Pipeline

In [ ]:
video_path = "../data/videos_160/videos_160/0.mp4"
processor = GolfVideoProcessor()

print("Processing video frames through MediaPipe + Savitzky-Golay + Hip/Torso Normalization...")
df = processor.process_video(video_path)
print(f"Processing complete! Extracted {len(df)} frames.")
df.head()

## 2. Inspect and Plot the Coordinates
Let's plot the raw vs. smoothed X and Y pixel coordinates of the right wrist to visualize the micro-jitter removal.

In [ ]:
plt.figure(figsize=(14, 6))

# Plot Wrist X coordinates
plt.subplot(1, 2, 1)
plt.plot(df["raw_right_wrist_x"], label="Raw Wrist X", color="red", alpha=0.6, linestyle="--")
plt.plot(df["smooth_right_wrist_x"], label="Smoothed Wrist X", color="green", linewidth=2)
plt.title("Right Wrist X Coordinate: Raw vs. Smoothed")
plt.xlabel("Frame Index")
plt.ylabel("Pixel Coordinate (X)")
plt.legend()
plt.grid(True)

# Plot Wrist Y coordinates
plt.subplot(1, 2, 2)
plt.plot(df["raw_right_wrist_y"], label="Raw Wrist Y", color="red", alpha=0.6, linestyle="--")
plt.plot(df["smooth_right_wrist_y"], label="Smoothed Wrist Y", color="green", linewidth=2)
plt.title("Right Wrist Y Coordinate: Raw vs. Smoothed")
plt.xlabel("Frame Index")
plt.ylabel("Pixel Coordinate (Y)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. Verify Normalization
Check the torso scale factor and verify that the mid-hip point is centered at (0, 0) in the normalized space.

In [ ]:
torso_scale = df["torso_scale"].iloc[0]
print(f"Calculated Torso Scale Factor: {torso_scale:.2f} pixels")

# Verify that the normalized mid-hip is indeed exactly (0,0)
norm_mid_hip_x = (df["norm_left_hip_x"] + df["norm_right_hip_x"]) / 2.0
norm_mid_hip_y = (df["norm_left_hip_y"] + df["norm_right_hip_y"]) / 2.0

print(f"Max normalized mid-hip X deviation: {np.abs(norm_mid_hip_x).max():.6e}")
print(f"Max normalized mid-hip Y deviation: {np.abs(norm_mid_hip_y).max():.6e}")

## 4. Generate Visual Jitter vs. Smooth Wrist Demo
Create a black canvas of the same dimensions as the original video. Draw the raw right wrist X/Y as a red dot/line, and the smoothed right wrist X/Y as a green dot/line.

In [ ]:
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)
demo_video_path = os.path.join(processed_dir, "wrist_demo.mp4")

width = int(df["width"].iloc[0])
height = int(df["height"].iloc[0])
fps = df["fps"].iloc[0]

# Open VideoWriter with mp4v codec
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(demo_video_path, fourcc, fps, (width, height))

raw_history = []
smooth_history = []

for idx, row in df.iterrows():
    # Create a blank black canvas
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    
    rx, ry = row["raw_right_wrist_x"], row["raw_right_wrist_y"]
    sx, sy = row["smooth_right_wrist_x"], row["smooth_right_wrist_y"]
    
    if not pd.isna(rx) and not pd.isna(ry):
        raw_history.append((int(rx), int(ry)))
    if not pd.isna(sx) and not pd.isna(sy):
        smooth_history.append((int(sx), int(sy)))
        
    # Draw historical raw trajectory (thin red lines)
    for i in range(1, len(raw_history)):
        cv2.line(frame, raw_history[i-1], raw_history[i], (0, 0, 150), 1)
        
    # Draw historical smoothed trajectory (medium green lines)
    for i in range(1, len(smooth_history)):
        cv2.line(frame, smooth_history[i-1], smooth_history[i], (0, 200, 0), 2)
        
    # Draw current positions as dots
    if not pd.isna(rx) and not pd.isna(ry):
        cv2.circle(frame, (int(rx), int(ry)), 6, (0, 0, 255), -1)
    if not pd.isna(sx) and not pd.isna(sy):
        cv2.circle(frame, (int(sx), int(sy)), 6, (0, 255, 0), -1)
        
    out.write(frame)

out.release()
print(f"Demo video successfully generated at: {demo_video_path}")

## 5. Generate Full Skeleton Overlays & Stick Figure Animations
Let's create two videos:
1. **`skeleton_overlay.mp4`**: The smoothed skeleton drawn directly over the original golf swing frames.
2. **`skeleton_blank.mp4`**: A minimalist, high-contrast stick figure animation on a pure black background.

In [7]:
from src.visualizer import draw_skeleton, POSE_CONNECTIONS, POSE_JOINTS

cap = cv2.VideoCapture(video_path)
overlay_out_path = os.path.join(processed_dir, 'skeleton_overlay.mp4')
blank_out_path = os.path.join(processed_dir, 'skeleton_blank.mp4')

out_overlay = cv2.VideoWriter(overlay_out_path, fourcc, fps, (width, height))
out_blank = cv2.VideoWriter(blank_out_path, fourcc, fps, (width, height))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret or frame_idx >= len(df):
        break
        
    row = df.iloc[frame_idx]
    
    # 1. Overlay frame
    overlay_frame = frame.copy()
    draw_skeleton(overlay_frame, row, prefix='smooth_', line_color=(0, 255, 0), joint_color=(0, 0, 255))
    
    # 2. Blank frame
    blank_frame = np.zeros((height, width, 3), dtype=np.uint8)
    draw_skeleton(blank_frame, row, prefix='smooth_', line_color=(255, 255, 255), joint_color=(0, 255, 0))
    
    out_overlay.write(overlay_frame)
    out_blank.write(blank_frame)
    frame_idx += 1

cap.release()
out_overlay.release()
out_blank.release()

print(f'Generated:\n - Overlay: {overlay_out_path}\n - Blank Stick Figure: {blank_out_path}')

Generated:
 - Overlay: ../data/processed/skeleton_overlay.mp4
 - Blank Stick Figure: ../data/processed/skeleton_blank.mp4


## 6. Save master processed dataset
Save the processed DataFrame to CSV.

In [8]:
output_csv_path = os.path.join(processed_dir, "video_0_processed.csv")
df.to_csv(output_csv_path, index=False)
print(f"Saved processed dataframe to: {output_csv_path}")

Saved processed dataframe to: ../data/processed/video_0_processed.csv
